# CS336 Spring 2026 - 13 Cross Entropy / AdamW / Learning Rate
----
> 12 已经得到 `(B,T,V)` logits。本章连接 loss、gradients 和 optimizer：

```text
logits → cross entropy → backward → clip gradients
       → AdamW step → learning-rate schedule
```

所有教学实现均有 PyTorch 数值参照或边界检查。


## 1. 从 logits 计算 Cross Entropy

单位置的 logits 为 $z\in\mathbb R^V$，正确类别为 $y$：

$$
\ell(z,y)=-\log\frac{e^{z_y}}{\sum_j e^{z_j}}
=\log\sum_j e^{z_j}-z_y.
$$

对 `(B,T,V)` logits 和 `(B,T)` 标签，要沿最后的 vocabulary 维计算概率，再对所有位置取 mean。


In [1]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(336)
torch.set_num_threads(2)


def stable_cross_entropy(logits, targets):
    if logits.shape[:-1] != targets.shape:
        raise ValueError("targets must match all non-vocab dimensions")
    shifted = logits - logits.max(dim=-1, keepdim=True).values
    chosen = shifted.gather(-1, targets.long().unsqueeze(-1)).squeeze(-1)
    return (torch.logsumexp(shifted, dim=-1) - chosen).mean()


B, T, V = 2, 5, 9
logits = torch.randn(B, T, V, requires_grad=True)
targets = torch.randint(0, V, (B, T))
ours = stable_cross_entropy(logits, targets)
reference = F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
torch.testing.assert_close(ours, reference)
print("Cross Entropy numerical comparison：通过")


Cross Entropy numerical comparison：通过


## 2. 为什么不能直接先 `exp(logits)`？

当 logits 很大时，`exp` 可能 overflow；当概率极小时，`log(softmax)` 可能 underflow。使用

$$
\log\sum_j e^{z_j}=m+\log\sum_j e^{z_j-m},\quad m=\max_jz_j
$$

可以提高数值稳定性。`torch.logsumexp` 已经实现了这一思想。


In [2]:
extreme = torch.tensor(
    [[1000.0, 999.0, -1000.0], [-1000.0, 1000.0, 1001.0]], requires_grad=True
)
labels = torch.tensor([0, 2])
extreme_loss = stable_cross_entropy(extreme, labels)
torch.testing.assert_close(extreme_loss, F.cross_entropy(extreme, labels))
extreme_loss.backward()
assert torch.isfinite(extreme_loss) and torch.isfinite(extreme.grad).all()
print("极端 logits 的 loss 和 grad 有限:", float(extreme_loss))


极端 logits 的 loss 和 grad 有限: 0.31326165795326233


/tmp/ipykernel_39389/2392963803.py:9: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("极端 logits 的 loss 和 grad 有限:", float(extreme_loss))


## 3. 验证梯度公式

单位置的交叉熵对 logits 的梯度为：

$$
\frac{\partial\ell}{\partial z_j}=p_j-\mathbf{1}(j=y).
$$

如果最后跨 N 个 token 位置求平均，还需整体除以 N。这里检查一个位置，让 autograd 与解析式对照。


In [3]:
z = torch.tensor([[0.2, -0.4, 1.2]], requires_grad=True)
target = torch.tensor([2])
stable_cross_entropy(z, target).backward()
formula = torch.softmax(z.detach(), dim=-1)
formula[0, target.item()] -= 1.0
torch.testing.assert_close(z.grad, formula, atol=1e-7, rtol=1e-6)
print("dCE/dlogits == softmax - one_hot：通过")


dCE/dlogits == softmax - one_hot：通过


## 4. AdamW 的动量、偏差修正与权重衰减

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,\qquad
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.
$$

$$
\hat m_t=m_t/(1-\beta_1^t),\qquad
\hat v_t=v_t/(1-\beta_2^t).
$$

AdamW 把参数衰减 $(1-\eta\lambda)\theta$ 与梯度的自适应更新分开。下面的单 Tensor 版本便于检查数学式，之后逐步对比 `torch.optim.AdamW`。


In [4]:
def teaching_adamw_step(param, grad, m, v, t, *, lr, beta1, beta2, eps, weight_decay):
    # 数学教学版本：不原地修改参数，返回更新后的参数和动量。
    m = beta1 * m + (1.0 - beta1) * grad
    v = beta2 * v + (1.0 - beta2) * grad.square()
    m_hat = m / (1.0 - beta1**t)
    v_hat = v / (1.0 - beta2**t)
    param = param * (1.0 - lr * weight_decay)
    param = param - lr * m_hat / (v_hat.sqrt() + eps)
    return param, m, v


settings = dict(lr=0.02, beta1=0.9, beta2=0.99, eps=1e-8, weight_decay=0.1)
initial = torch.tensor([1.0, -1.5, 0.5], dtype=torch.float64)
p_manual = initial.clone()
m, v = torch.zeros_like(initial), torch.zeros_like(initial)
p_torch = nn.Parameter(initial.clone())
optimizer = torch.optim.AdamW(
    [p_torch],
    lr=settings["lr"],
    betas=(settings["beta1"], settings["beta2"]),
    eps=settings["eps"],
    weight_decay=settings["weight_decay"],
    foreach=False,
)

gradients = [
    torch.tensor([0.4, -0.3, 0.1], dtype=torch.float64),
    torch.tensor([-0.2, 0.5, 0.8], dtype=torch.float64),
    torch.tensor([0.0, 0.1, -0.3], dtype=torch.float64),
]
for t, grad in enumerate(gradients, start=1):
    p_manual, m, v = teaching_adamw_step(p_manual, grad, m, v, t, **settings)
    optimizer.zero_grad(set_to_none=True)
    p_torch.grad = grad.clone()
    optimizer.step()
    torch.testing.assert_close(p_manual, p_torch.detach(), atol=1e-10, rtol=1e-10)

print("3 步 AdamW 教学更新 == torch.optim.AdamW：通过")


3 步 AdamW 教学更新 == torch.optim.AdamW：通过


## 5. AdamW 不等于直接往 Adam 的梯度里加入 L2

Adam + L2 的正则项会进入一阶/二阶动量；AdamW 的 weight decay 则与自适应分母解耦。

下面在相同初值和梯度上比较两者，不用文字争论，直接观察结果。


In [5]:
p_adam = nn.Parameter(initial.clone())
p_adamw = nn.Parameter(initial.clone())
adam = torch.optim.Adam([p_adam], lr=0.02, weight_decay=0.1, foreach=False)
adamw = torch.optim.AdamW([p_adamw], lr=0.02, weight_decay=0.1, foreach=False)
for p in (p_adam, p_adamw):
    p.grad = torch.tensor([0.4, 0.1, -0.2], dtype=torch.float64)
adam.step()
adamw.step()
assert not torch.allclose(p_adam.detach(), p_adamw.detach())
print("Adam + L2:", p_adam.detach().tolist())
print("AdamW:", p_adamw.detach().tolist())


Adam + L2: [0.9800000004, -1.4800000039999992, 0.5199999986666668]
AdamW: [0.9780000005, -1.5169999980000002, 0.518999999]


## 6. Warmup + Cosine Decay

$$
\eta(t)=
\begin{cases}
\eta_{\max}t/T_w & t<T_w\\
\eta_{\min}+\frac{\eta_{\max}-\eta_{\min}}2
(1+\cos(\pi(t-T_w)/(T_c-T_w))) & T_w\leq t\leq T_c\\
\eta_{\min} & t>T_c .
\end{cases}
$$

这里约定 `t=0` 时 learning rate 为 0，`t=T_w` 到达 max_lr，`t=T_c` 到达 min_lr。不同训练框架可能使用不同 step 编号约定。


In [6]:
def cosine_lr(step, max_lr, min_lr, warmup_steps, decay_steps):
    if warmup_steps < 0 or decay_steps <= warmup_steps or step < 0:
        raise ValueError("invalid schedule range")
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step > decay_steps:
        return min_lr
    ratio = (step - warmup_steps) / (decay_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * ratio))


values = [(t, cosine_lr(t, 0.01, 0.001, 4, 16)) for t in [0, 1, 4, 10, 16, 20]]
assert values[0][1] == 0
assert math.isclose(values[2][1], 0.01)
assert math.isclose(values[4][1], 0.001)
assert math.isclose(values[5][1], 0.001)
print("step, learning-rate:", values)


step, learning-rate: [(0, 0.0), (1, 0.0025), (4, 0.010000000000000002), (10, 0.0055000000000000005), (16, 0.001), (20, 0.001)]


## 7. Gradient Clipping 作用于全局梯度范数

全局 L2 范数大于阈值 $c$ 时，让**所有**参数的梯度乘同一个系数：

$$
g \leftarrow g\cdot\min(1,c/\|g\|_2).
$$

这不同于逐元素 clamp。通常顺序为：

```text
zero_grad → forward → backward → clip_grad_norm_ → optimizer.step
```


In [7]:
layer = nn.Linear(4, 3)
layer(torch.randn(5, 4) * 100).square().mean().backward()
params = list(layer.parameters())


def total_grad_norm(params):
    grads = [p.grad.detach().reshape(-1) for p in params if p.grad is not None]
    return torch.cat(grads).norm(2) if grads else torch.tensor(0.0)


before = total_grad_norm(params)
reported = nn.utils.clip_grad_norm_(params, max_norm=0.5)
after = total_grad_norm(params)
torch.testing.assert_close(before, reported)
assert after.item() <= 0.50001
print("before clip:", float(before), "| after clip:", float(after))


before clip: 9545.6650390625 | after clip: 0.5


## 8. 交叉熵与 Perplexity

使用自然对数的 mean token negative log-likelihood（NLL）时：

$$
\mathrm{Perplexity}=\exp(\mathrm{mean\ NLL}).
$$

不同 tokenizer 的 token 粒度不同，因此不能脱离评测协议直接比较 token-level perplexity。


In [8]:
mean_nll = float(F.cross_entropy(torch.randn(6, 10), torch.randint(0, 10, (6,))))
ppl = math.exp(mean_nll)
assert math.isfinite(ppl) and ppl > 0
print(
    "illustrative mean NLL:",
    round(mean_nll, 4),
    "| illustrative perplexity:",
    round(ppl, 4),
)


illustrative mean NLL: 3.2292 | illustrative perplexity: 25.2598


## 9. 小结 → 14 Training Loop / Checkpoints / Sampling

完成的训练侧基本序列：

```text
logits + labels → stable loss → backward
               → gradient clipping → AdamW
               → learning-rate schedule
```

接下来补上 tokenized data、batch 切片、optimizer state 存取以及生成过程。
